# 04 Terminal 8k Jackknife

This notebook is the fourth evidence notebook in the public release suite.

## Purpose

It reproduces the **terminal 8k jackknife robustness audit** and shows that:

- the terminal 8k `h1` lock is not driven by a tiny outlier subset
- the concentration remains extremely stable under repeated packet subsampling
- the release-level robustness claim is supported by a large jackknife envelope

This notebook is **artifact-first**. By default it loads frozen jackknife outputs and rebuilds the headline summary table and release plots without rerunning the 10,000-iteration subsampling audit.


## Reading note

This notebook is the final main evidence notebook in the public release suite.

It supports the claim that the terminal 8k lock is **systemic across the packet population**, not merely carried by a few dominant packets.


In [ ]:
# Optional path settings for Colab or local runs

import os
import json
from pathlib import Path

DEFAULT_OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/ForgeV16c"
OUT_DIR = os.environ.get("FORGE_V16C_OUT_DIR", DEFAULT_OUT_DIR)

print("OUT_DIR =", OUT_DIR)

## Expected frozen artifacts

This notebook looks for the following terminal 8k jackknife artifacts:

- `s7_jackknife_results_v16c_*.csv`  
  or a specific frozen results CSV if you rename it into the release folder

- `s7_jackknife_histogram_v16c_*.png`  
  optional, not required for the notebook logic

If you prefer stable filenames, point `RESULTS_CSV` below to the exact release file you want to use.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Point this to a single frozen CSV if you prefer a fixed release filename.
RESULTS_CSV = None

if RESULTS_CSV is None:
    matches = sorted(glob.glob(os.path.join(OUT_DIR, "s7_jackknife_results_v16c_*.csv")))
    RESULTS_CSV = matches[-1] if matches else None

print("RESULTS_CSV =", RESULTS_CSV)
print("FOUND" if RESULTS_CSV and os.path.exists(RESULTS_CSV) else "MISSING")

## Load the frozen jackknife results

The expected CSV contains two columns:

- `Resultant_R`
- `Arc80`

Each row corresponds to one subsampling iteration of the frozen terminal 8k packet field.


In [ ]:
if RESULTS_CSV is None or not os.path.exists(RESULTS_CSV):
    raise FileNotFoundError("Could not locate a terminal 8k jackknife results CSV. Set RESULTS_CSV manually.")

jk_df = pd.read_csv(RESULTS_CSV)
display(jk_df.head())
print("Iterations loaded:", len(jk_df))

## Build the release summary

This cell computes the headline summary statistics used in the paper:

- mean `Resultant_R`
- 1st percentile `Resultant_R`
- worst-case `Resultant_R`
- mean `Arc80`
- 99th percentile `Arc80`
- worst-case `Arc80`


In [ ]:
summary = {
    "iterations": int(len(jk_df)),
    "r_mean": float(jk_df["Resultant_R"].mean()),
    "r_01": float(np.percentile(jk_df["Resultant_R"], 1)),
    "r_min": float(jk_df["Resultant_R"].min()),
    "arc_mean": float(jk_df["Arc80"].mean()),
    "arc_99": float(np.percentile(jk_df["Arc80"], 99)),
    "arc_max": float(jk_df["Arc80"].max()),
}

summary_df = pd.DataFrame([summary])
display(summary_df)

## Headline interpretation

The jackknife audit repeatedly removes a fixed fraction of packets from the terminal 8k field and recomputes the concentration statistics.

The release-level interpretation is:

- if the phase lock were carried by only a tiny set of packets, the subsampled distributions would spread out sharply
- instead, both `Resultant_R` and `Arc80` remain extremely tight across the subsampling ensemble

That is the empirical signature of a **distributed**, not merely **outlier-driven**, lock.


In [ ]:
headline = pd.DataFrame([{
    "iterations": summary["iterations"],
    "r_mean": summary["r_mean"],
    "r_01": summary["r_01"],
    "r_min": summary["r_min"],
    "arc_mean": summary["arc_mean"],
    "arc_99": summary["arc_99"],
    "arc_max": summary["arc_max"],
}])

display(headline)

## Plot the jackknife distributions

Panel A shows the distribution of subsampled `Resultant_R`.  
Panel B shows the distribution of subsampled `Arc80`.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(jk_df["Resultant_R"], bins=50, alpha=0.8)
ax.axvline(summary["r_01"], linestyle="--", linewidth=2, label=f"1st pctile = {summary['r_01']:.6f}")
ax.axvline(summary["r_min"], linestyle=":", linewidth=2, label=f"worst = {summary['r_min']:.6f}")
ax.set_title("A. Jackknife Resultant R distribution")
ax.set_xlabel("Resultant R")
ax.set_ylabel("Frequency")
ax.legend()

ax = axes[1]
ax.hist(jk_df["Arc80"], bins=50, alpha=0.8)
ax.axvline(summary["arc_99"], linestyle="--", linewidth=2, label=f"99th pctile = {summary['arc_99']:.6f}")
ax.axvline(summary["arc_max"], linestyle=":", linewidth=2, label=f"worst = {summary['arc_max']:.6f}")
ax.set_title("B. Jackknife Arc80 distribution")
ax.set_xlabel("Arc80")
ax.set_ylabel("Frequency")
ax.legend()

plt.tight_layout()
plt.show()

## Simple release checks

These are the release-level checks this notebook should satisfy:

- the jackknife mean `Resultant_R` remains extremely high
- the 1st percentile `Resultant_R` remains close to the full-field value
- the worst-case `Arc80` remains tightly bounded
- the distributions remain narrow rather than exploding under subsampling


In [ ]:
checks = {
    "iterations_ge_10000": summary["iterations"] >= 10000,
    "r_mean_gt_0p999": summary["r_mean"] > 0.999,
    "r_01_gt_0p999": summary["r_01"] > 0.999,
    "arc_99_lt_0p016": summary["arc_99"] < 0.016,
    "arc_max_lt_0p016": summary["arc_max"] < 0.016,
}

checks_df = pd.DataFrame([checks])
display(checks_df)

if checks_df.all(axis=None):
    print("All terminal 8k jackknife release checks passed.")
else:
    print("One or more terminal 8k jackknife release checks failed. Inspect the frozen artifact.")

## Optional export

If you want to save the compact release summary into the repository artifacts folder, run the next cell.


In [ ]:
# Optional export
# export_df = summary_df.copy()
# export_path = os.path.join(OUT_DIR, "release_terminal8k_jackknife_summary.csv")
# export_df.to_csv(export_path, index=False)
# print("Saved:", export_path)

## Heavy recomputation note

The 10,000-iteration jackknife is intentionally **not rerun by default** in this public notebook.  
This notebook is meant to read and explain the frozen release artifact.

If you later decide to expose the heavy rerun path, add it as a clearly marked optional section at the end of the notebook.


## Release suite complete

At this point, the public release suite contains:

- `00_release_guide.ipynb`
- `01_broad_board_ablation.ipynb`
- `02_terminal_ladder.ipynb`
- `03_terminal_8k_matched_packet_null.ipynb`
- `04_terminal_8k_jackknife.ipynb`

This is the compact public-facing evidence path corresponding to the current frozen paper release.
